# Program 4 – Train a Deep MLP for Multi-Class Classification

**Objective:** Build a deep MLP and fine-tune it using Dropout and Batch Normalization.

**Dataset:** Fashion-MNIST (70,000 grayscale 28×28 images of clothing, 10 classes)

**Key tasks**
1. Use 3–5 dense layers with ReLU + Softmax output.
2. Apply the Adam optimizer and evaluate training / test accuracy.
3. Show training curves.

## Theory (for viva)

- **Deep MLP:** several fully-connected (Dense) hidden layers; each computes $h = f(Wx + b)$.
- **ReLU:** $f(x) = \max(0, x)$ — cheap, avoids vanishing gradient for positive inputs.
- **Softmax output:** converts 10 scores into class probabilities.
- **Batch Normalization:** normalizes each layer's pre-activations over the mini-batch to mean 0 / variance 1, then scales and shifts with learnable $\gamma, \beta$:
  $\hat x = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}},\ y = \gamma \hat x + \beta$. Gives faster, more stable training and allows larger learning rates.
- **Dropout(p):** during training randomly sets a fraction $p$ of neurons to 0 each step → network cannot rely on single neurons → less overfitting. Disabled at test time.
- **Order used here:** `Dense → BatchNorm → ReLU → Dropout` (Dense has `use_bias=False` because BN's $\beta$ already acts as a bias).
- **Adam:** adaptive-learning-rate optimizer with momentum.

## Step 1 – Import libraries

In [ ]:
import keras  # Keras 3 API (runs on TensorFlow backend)
from keras import layers  # Layer classes (Dense, Dropout, Embedding, ...)
import matplotlib.pyplot as plt  # Plotting graphs
import numpy as np  # Numerical arrays
import tensorflow as tf  # TensorFlow (tensors, GradientTape, seeds)

tf.random.set_seed(42)                  # Reproducible weight initialization
np.random.seed(42)                      # Reproducible NumPy randomness

## Step 2 – Load and preprocess Fashion-MNIST

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()  # Load dataset

class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",           # Names of the 10 classes
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]               # (index = label value)

x_train = x_train.reshape(-1, 784).astype("float32") / 255.0   # Flatten images and scale to [0, 1]
x_test = x_test.reshape(-1, 784).astype("float32") / 255.0     # Same for test

print("Train:", x_train.shape, "Test:", x_test.shape)           # (60000, 784) (10000, 784)

In [ ]:
plt.figure(figsize=(10, 4))                                     # Figure for sample images
for i in range(10):                                             # Show first 10 training images
    plt.subplot(2, 5, i + 1)                                    # 2 rows x 5 columns grid
    plt.imshow(x_train[i].reshape(28, 28), cmap="gray")         # Reshape back to 28x28 for display
    plt.title(class_names[y_train[i]])                          # Class name as title
    plt.axis("off")                                             # Hide axes
plt.tight_layout()                                              # Tidy layout
plt.show()                                                      # Display

## Step 3 – Build the deep MLP (4 hidden Dense layers + BN + Dropout)

In [ ]:
def dense_block(units, dropout_rate):                           # Helper that returns one hidden block as a list of layers
    return [
        layers.Dense(units, use_bias=False),                    # Linear layer (bias not needed before BN)
        layers.BatchNormalization(),                            # Normalize activations over the batch
        layers.Activation("relu"),                              # ReLU non-linearity
        layers.Dropout(dropout_rate),                           # Randomly drop neurons during training
    ]

model = keras.Sequential(                                       # Build model layer by layer
    [keras.Input(shape=(784,))]                                 # Input: 784 pixels
    + dense_block(512, 0.3)                                     # Hidden layer 1: 512 units, 30% dropout
    + dense_block(256, 0.3)                                     # Hidden layer 2: 256 units, 30% dropout
    + dense_block(128, 0.2)                                     # Hidden layer 3: 128 units, 20% dropout
    + dense_block(64, 0.2)                                      # Hidden layer 4: 64 units, 20% dropout
    + [layers.Dense(10, activation="softmax")]                  # Output layer: 10 class probabilities
)

model.summary()                                                 # Print architecture and parameter counts

## Step 4 – Compile with Adam

In [ ]:
model.compile(                                                  # Configure training
    optimizer=keras.optimizers.Adam(learning_rate=0.001),       # Adam optimizer, lr = 0.001
    loss="sparse_categorical_crossentropy",                     # Cross-entropy with integer labels
    metrics=["accuracy"],                                       # Track accuracy
)

## Step 5 – Train (with EarlyStopping + learning-rate reduction for fine-tuning)

In [ ]:
callbacks = [                                                   # Callbacks run automatically during training
    keras.callbacks.EarlyStopping(monitor="val_loss",           # Watch validation loss
                                  patience=5,                   # Stop if no improvement for 5 epochs
                                  restore_best_weights=True),   # Roll back to best epoch's weights
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss",       # Watch validation loss
                                      factor=0.5,               # Halve the learning rate ...
                                      patience=2,               # ... if no improvement for 2 epochs
                                      min_lr=1e-5),             # Never go below 1e-5
]

history = model.fit(                                            # Start training
    x_train, y_train,                                           # Training data
    epochs=30,                                                  # Maximum epochs (early stopping may end sooner)
    batch_size=128,                                             # Mini-batch size
    validation_split=0.1,                                       # 10% of training data for validation
    callbacks=callbacks,                                        # Use the callbacks above
    verbose=2,                                                  # One line per epoch
)

## Step 6 – Evaluate training and test accuracy

In [ ]:
train_loss, train_acc = model.evaluate(x_train, y_train, verbose=0)  # Accuracy on training set (dropout OFF)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)      # Accuracy on unseen test set

print(f"Training accuracy : {train_acc:.4f}  (loss {train_loss:.4f})")  # Print train result
print(f"Test accuracy     : {test_acc:.4f}  (loss {test_loss:.4f})")    # Print test result
print(f"Generalization gap: {train_acc - test_acc:.4f}")                # Small gap = little overfitting

## Step 7 – Training curves

In [ ]:
hist = history.history                                          # Dict with per-epoch metrics
epochs_range = range(1, len(hist["loss"]) + 1)                  # X-axis: epoch numbers

fig, axes = plt.subplots(1, 2, figsize=(14, 5))                 # Two plots side by side

axes[0].plot(epochs_range, hist["loss"], "o-", label="Train loss")        # Training loss curve
axes[0].plot(epochs_range, hist["val_loss"], "s-", label="Validation loss")  # Validation loss curve
axes[0].set_title("Loss vs Epochs")                             # Title
axes[0].set_xlabel("Epoch")                                     # X label
axes[0].set_ylabel("Loss")                                      # Y label
axes[0].legend()                                                # Legend
axes[0].grid(True)                                              # Grid

axes[1].plot(epochs_range, hist["accuracy"], "o-", label="Train accuracy")        # Training accuracy curve
axes[1].plot(epochs_range, hist["val_accuracy"], "s-", label="Validation accuracy")  # Validation accuracy curve
axes[1].set_title("Accuracy vs Epochs")                         # Title
axes[1].set_xlabel("Epoch")                                     # X label
axes[1].set_ylabel("Accuracy")                                  # Y label
axes[1].legend()                                                # Legend
axes[1].grid(True)                                              # Grid

plt.tight_layout()                                              # Tidy layout
plt.show()                                                      # Show figure

## Step 8 – Sample predictions

In [ ]:
probs = model.predict(x_test[:15], verbose=0)                   # Predicted probabilities for 15 test images
preds = np.argmax(probs, axis=1)                                # Predicted class = index of max probability

plt.figure(figsize=(15, 6))                                     # Figure
for i in range(15):                                             # Loop over the 15 images
    plt.subplot(3, 5, i + 1)                                    # 3 x 5 grid
    plt.imshow(x_test[i].reshape(28, 28), cmap="gray")          # Show image
    color = "green" if preds[i] == y_test[i] else "red"         # Green = correct, red = wrong
    plt.title(f"P: {class_names[preds[i]]}\nT: {class_names[y_test[i]]}", color=color, fontsize=9)  # Pred vs true
    plt.axis("off")                                             # Hide axes
plt.tight_layout()                                              # Tidy layout
plt.show()                                                      # Display

## Expected observations
- Test accuracy around **88–90%** on Fashion-MNIST.
- Train and validation curves stay close → Dropout + BatchNorm keep overfitting low.
- Validation accuracy may be slightly *higher* than training accuracy early on, because dropout is active only during training.

## Viva questions
1. Why is dropout turned off at test time? → We want deterministic predictions using the full network; Keras scales activations during training (inverted dropout) so no change is needed at test time.
2. Why `use_bias=False` before BatchNorm? → BN subtracts the mean (cancels the bias) and adds its own learnable shift $\beta$.
3. What does BN do at inference? → Uses running (moving) mean and variance collected during training, not batch statistics.
4. What does `restore_best_weights=True` do? → Keeps the weights from the epoch with the best validation loss.
5. Why softmax in the output layer? → Multi-class problem; gives a probability distribution over 10 classes.